# Job offers: preprocessing

First half of the job-offer build, everything that does not need the KB mapping: normalize the raw
export, drop duplicates and offers without skills, assign `id_j`, and write the input of the five
mapping notebooks.

- Normalize the minimum-experience and education labels (Spanish to EQF levels and English year
  buckets) and parse the raw skill string into a set.
- Drop exact duplicates, then offers with no skill. `id_j` is then assigned as 0..N-1.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import re
import pandas as pd

In [ ]:
# Minimum-experience labels to English year buckets.
min_experience = {
    "Sólo prácticas": "1 - 2 years",
    "De 1 a 2 años": "1 - 2 years",
    "De 2 a 3 años": "2 - 3 years",
    "De 3 a 5 años": "3 - 5 years",
    "De 5 a 10 años": "5 - 10 years",
}

# Education labels to the closest European Qualifications Framework (EQF) level.
educationLevel = {
    "Bachillerato": "level 4",
    "Sin estudios": "level 0",
    "Grado": "level 6",
    "Diplomatura": "level 6",
    "Licenciatura": "level 7",
    "Máster": "level 7",
    "Postgrado": "level 7",
    "Doctorado": "level 8",
    "Ingeniería Técnica": "level 6",
    "Ingeniería Superior": "level 7",
    "Educación Secundaria Obligatoria": "level 4",
    "Formación Profesional Grado Superior": "level 5",
    "Ciclo Formativo Grado Superior": "level 5",
    "Formación Profesional Grado Medio": "level 5",
    "Ciclo Formativo Grado Medio": "level 5",
    "Enseñanzas artísticas (regladas)": "level 5",
    "Enseñanzas deportivas (regladas)": "level 5",
    "Otros títulos, certificaciones y carnés": "level 0",
    "Otros cursos y formación no reglada": "level 0"
}

In [ ]:
def clean_skill_list(val: str):
    """Parse the raw skill string into a tuple of unique, cleaned labels in a fixed order."""
    res = re.split("|".join(("'\n '", r"'\s'")), val)
    clean_res = [r.replace("'", '').replace('[', '').replace(']', '') for r in res]
    return tuple(sorted(set(clean_res)))

In [ ]:
jobs_dt = pd.read_csv(
    "../source/offers_examples_10k.csv",
    usecols=["job_title", "description", "skills_list", "studies_level_min", "min_experience"],
    converters={
        "job_title": lambda x : str(x).strip().lower(),
        "description" : lambda x : str(x).strip().lower(),
        "skills_list": clean_skill_list,
        "min_experience": lambda x: min_experience.get(x,"Not specified"),
        "studies_level_min": lambda  x: educationLevel.get(x, "level 0")
    }
)
jobs_dt.head(10)

In [ ]:
before = len(jobs_dt)
jobs_dt = jobs_dt.drop_duplicates().reset_index(drop=True)
print(f"offers: {before} -> {len(jobs_dt)} ({before - len(jobs_dt)} duplicates)")

In [ ]:
before = len(jobs_dt)
jobs_dt = jobs_dt[jobs_dt.skills_list.str.len() > 0].reset_index(drop=True)
print(f"offers: {before} -> {len(jobs_dt)} ({before - len(jobs_dt)} with no skill)")

In [ ]:
jobs_dt["id_j"] = [*range(len(jobs_dt))]

# Input of the five mapping notebooks, which key their output on id_j.
jobs_dt[["id_j", "job_title", "description", "skills_list"]].to_json(
    "mapping/preprocessed_jobs.json", orient="records", indent=2)

# stage1 keeps only what 6_build_jobs.ipynb needs from the raw export; the texts come back
# translated.
jobs_dt = jobs_dt.drop(columns=["job_title", "skills_list", "description"])
jobs_dt.to_json("mapping/stage1_jobs.json", orient="records", indent=2)
print(f"{len(jobs_dt)} offers -> mapping/preprocessed_jobs.json + mapping/stage1_jobs.json")